In [1]:
from Models.EnergyStorageModel import EnergyStorageModel as ESM
import pandas as pd 
import numpy as np 
import plotly.express as px
import plotly.graph_objects as go

In [2]:
policy_train_files = {
    0.0: "./Data/policy_train.csv",
    0.15: "./Data/policy_train_all_feat_noise015.csv",
    0.25: "./Data/policy_train_all_feat_noise025.csv",
    0.35: "./Data/policy_train_all_feat_noise035.csv"
}

policy_test_files = {
    0.0: "./Data/policy_test.csv",
    0.15: "./Data/policy_test_all_feat_noise015.csv",
    0.25: "./Data/policy_test_all_feat_noise025.csv",
    0.35: "./Data/policy_test_all_feat_noise035.csv"
}

def load_and_reshape(file_path, keep_first_column=False):
    df = pd.read_csv(file_path)
    
    if not keep_first_column:
        df.drop(columns=["0"], inplace=True)
    
    array = df.to_numpy()
    num_cols = array.shape[1]
    new_length = (array.shape[0] // 24) * 24
    array = array[:new_length, :]
    reshaped_array = array.reshape(new_length // 24, 24, num_cols).transpose(2, 0, 1)
    
    return reshaped_array

In [3]:
hist_prices = load_and_reshape(policy_test_files[0.35], keep_first_column=True)

In [4]:
test_data = hist_prices[0, 3:, :]
train_data = hist_prices[1:, :3, :]

In [5]:
# test_data = hist_prices[0, 3:240, :]
# train_data = hist_prices[1:, :3, :]
# chosen_train_scenarios = np.random.choice(train_data.shape[0], 10, replace=False)


# train_scen = [train_data[idx, :5, :] for idx in chosen_train_scenarios]

# sampled_means = []

# for i in range(5):
#     sampled_indices = np.random.choice(len(train_data), 10, replace=False)
#     sampled_values = [train_data[idx] for idx in sampled_indices]
#     mean_values = np.mean(sampled_values, axis=0)
#     sampled_means.append(mean_values)

# sampled_means = np.array(sampled_means)


# train_scen_stack = np.stack(train_scen, axis=0)

In [6]:
# train_scen_mean = np.mean(train_scen_stack, axis=0)

In [7]:
# fig = go.Figure()

# fig.add_trace(go.Scatter(y=test_data[0, :], mode="lines", name="Test Scenario"))
# fig.add_trace(go.Scatter(y=train_scen_mean[0, :], mode="lines", name="Train Mean"))
# for i in range(train_scen_stack.shape[0]):
#     fig.add_trace(go.Scatter(y=train_scen_stack[i, 0, :], mode="lines", name=f"Train Scenario {i}"))
# fig.show()

In [8]:
test_data = hist_prices[0, :, :]

initial_state = {"energy_amount": 300, "price": test_data[0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "norm"
exog_params = {"hist_price": test_data[1:]}
T = len(test_data[1:])
t0 = 0

model = ESM(
    t0=t0,
    T=T,
    seed=0,
    init_args=init_args,
    exog_params=exog_params,
    S0=initial_state,
    model_name=model_name
)

In [ ]:
from Models.Policies.DLA import DeterministicLookahead
from Models.Policies.PFA import BuyLowSellHigh
# from Models.EnergyStoragePolicy import BADP
from Models.Policies.VFA import BADP

# policy = BuyLowSellHigh(model=model, policy_name="blsh", verbose=True, theta_low=100, theta_high=105)
policy = DeterministicLookahead(model=model, policy_name="lookahead", horizon=4, verbose=False, benchmark=False)
# policy = BADP(
#     model=model,
#     policy_name="badp", 
#     price_samples=train_scen_stack,
#     # price_samples=hist_prices[0, :3, :], 
#     sample_size=3,
#     discount_factor= 0.99, 
#     test_size=0.1, 
#     verbose=True,
#     aggregation_method="combine",
#     model_type="linear",
#     energy_bonus_factor=0.08,
#     )


# h=2 14042.7447368421 -> best h for benchmark


In [13]:
policy.run_policy(n_iterations=1)

12532.958559210521

In [12]:
# print(f"{policy.results.tail(1)["C_t sum"].item():.2f}")

In [13]:
# from Models.BaseClasses.Util import grid_search

# grid = {
    
# }

# result = grid_search(policy=policy, grid=grid, n_iterations=1, ordered=True)
# print(f"Best parameters: {result['best_parameters']} with an objective of {result['best_performance']}.")

In [14]:
# res_grid = result["all_runs"].pivot(index="theta_low", columns="theta_high", values="performance")
# px.imshow(res_grid)

In [15]:
# result["all_runs"].sort_values("performance", ascending=False)